# 🔍 IEEE-CIS Fraud Detection
## Detailed EDA and Preprocessing Notebook

This notebook follows the project guidelines and focuses on the early data science stages:
- Data loading and validation
- Exploratory Data Analysis (EDA)
- Data cleaning and preprocessing
- Feature engineering and split preparation
- Saved artifacts for the three models: Decision Tree, XGBoost, and HGNN

The notebook is designed to be reproducible and suitable for coursework submission.

In [5]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pickle
import joblib

PROJECT_ROOT = Path('/Users/aaronr/Desktop/PROJECT_P')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import *
from src.utils import get_logger, set_plot_style, format_number, format_pct
from src.data_loader import load_raw_data, get_dataset_summary
from src.preprocessing import (
    handle_missing_values,
    cap_outliers,
    encode_categoricals,
    split_data,
    scale_features,
    run_preprocessing_pipeline,
)

logger = get_logger('Preprocessing-EDA')
set_plot_style()

RANDOM_SEED = RANDOM_STATE
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

SAMPLE_FRAC = 0.3
OUTPUT_DIR = Path(PROJECT_ROOT) / 'outputs'
MODEL_OUT_DIR = Path(PROJECT_ROOT) / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Sample frac:  {SAMPLE_FRAC}')
print(f'Random seed:  {RANDOM_SEED}')

Project root: /Users/aaronr/Desktop/PROJECT_P
Output dir:   /Users/aaronr/Desktop/PROJECT_P/outputs
Sample frac:  0.3
Random seed:  42


## 1. Data Loading and Basic Validation

This section loads the merged IEEE-CIS fraud dataset, previews the schema, and checks the first-level quality conditions:
- non-empty file contents
- required target and ID columns
- duplicate rows
- general shape and target prevalence

In [6]:
from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype

# Load raw merged data
try:
    df = load_raw_data(sample_frac=SAMPLE_FRAC)
except Exception as exc:
    raise RuntimeError(f'Could not load raw data: {exc}')

summary = get_dataset_summary(df)

required_cols = [TARGET_COL, ID_COL]
missing_required = [c for c in required_cols if c not in df.columns]
assert not missing_required, f'Missing required columns: {missing_required}'
assert len(df) > 0, 'Loaded dataframe is empty.'

print(f'Dataset shape: {df.shape}')
print(f'Fraud rate: {format_pct(summary["fraud_rate"])}')
print(f'Imbalance ratio: 1:{summary["imbalance_ratio"]:.1f}')
print(f'Duplicate rows: {df.duplicated().sum():,}')
print('\nPreview:')
display(df.head(5))

schema_preview = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing_pct': df.isna().mean().values,
    'n_unique': [df[c].nunique(dropna=True) for c in df.columns],
})
print('\nSchema preview:')
display(schema_preview.head(15))

20:26:06 | DataLoader           | INFO    | ⏳ Starting: Loading transaction data
20:26:11 | DataLoader           | INFO    |   Transactions: 590,540 rows × 394 cols
20:26:11 | DataLoader           | INFO    | ✅ Finished: Loading transaction data (4.5s)
20:26:11 | DataLoader           | INFO    | ⏳ Starting: Loading identity data
20:26:11 | DataLoader           | INFO    |   Identity:     144,233 rows × 41 cols
20:26:11 | DataLoader           | INFO    | ✅ Finished: Loading identity data (0.2s)
20:26:11 | DataLoader           | INFO    | ⏳ Starting: Merging datasets on TransactionID (LEFT JOIN)
20:26:11 | DataLoader           | INFO    |   Merged:       590,540 rows × 434 cols
20:26:11 | DataLoader           | INFO    |   Identity coverage: 24.42% of transactions
20:26:11 | DataLoader           | INFO    | ✅ Finished: Merging datasets on TransactionID (LEFT JOIN) (0.1s)
20:26:11 | DataLoader           | INFO    | ⏳ Starting: Optimizing memory usage
20:26:13 | DataLoader           | INFO

  Memory: 1984.2 MB → 1073.5 MB (45.9% reduction)


20:26:13 | DataLoader           | INFO    |   Sampled: 590,540 → 177,162 rows (30.00%)
20:26:13 | DataLoader           | INFO    | ============================================================
20:26:13 | DataLoader           | INFO    |          DATASET SUMMARY
20:26:13 | DataLoader           | INFO    | ============================================================
20:26:13 | DataLoader           | INFO    |   Total samples:     177,162
20:26:13 | DataLoader           | INFO    |   Total features:    434
20:26:13 | DataLoader           | INFO    |   Fraud (1):         6,341 (3.58%)
20:26:13 | DataLoader           | INFO    |   Legitimate (0):    170,821 (96.42%)
20:26:13 | DataLoader           | INFO    |   Imbalance ratio:   1:26.9
20:26:13 | DataLoader           | INFO    |   Numeric features:  403
20:26:13 | DataLoader           | INFO    |   Categorical:       31
20:26:13 | DataLoader           | INFO    |   Cols no missing:   20
20:26:13 | DataLoader           | INFO    |   Cols hig

Dataset shape: (177162, 434)
Fraud rate: 3.58%
Imbalance ratio: 1:26.9
Duplicate rows: 0

Preview:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,3457624,0,12153579,724.000000,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3552820,0,15005886,108.500000,W,12544,321.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3271083,0,6970178,47.950001,W,9400,111.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3226689,0,5673658,100.598999,C,15885,545.0,185.0,visa,138.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3268855,0,6886780,107.949997,W,15497,490.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Schema preview:


,column,dtype,missing_pct,n_unique
0,TransactionID,int32,0.000000,177162
1,isFraud,int8,0.000000,2
2,TransactionDT,int32,0.000000,175575
3,TransactionAmt,float32,0.000000,10724
4,ProductCD,str,0.000000,5
5,card1,int16,0.000000,9475
6,card2,float32,0.014800,500
7,card3,float32,0.002596,95
8,card4,str,0.002613,4
9,card5,float32,0.007027,95


## 3. Initial Data Quality Audit

This section summarizes missingness, duplicate counts, high-cardinality categorical features, and a compact quality report for the full dataset.

In [8]:
missingness = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': df.isna().mean(),
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(dropna=True),
}).sort_values(['missing_pct', 'n_unique'], ascending=[False, False])

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

high_missing = missingness[missingness['missing_pct'] >= 0.5]
cardinality = missingness[missingness['dtype'].isin(['object', 'category'])].sort_values('n_unique', ascending=False)

quality_report = pd.DataFrame({
    'metric': [
        'rows', 'columns', 'duplicate_rows', 'target_missing', 'high_missing_columns',
        'numeric_columns', 'categorical_columns'
    ],
    'value': [
        len(df),
        df.shape[1],
        int(df.duplicated().sum()),
        int(df[TARGET_COL].isna().sum()) if TARGET_COL in df.columns else np.nan,
        int(len(high_missing)),
        int(len(numeric_cols)),
        int(len(categorical_cols)),
    ]
})

print('Quality report:')
display(quality_report)

print('Top missing columns:')
display(missingness.head(20))

print('High-cardinality categorical columns:')
display(cardinality.head(20))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
missingness['missing_pct'].head(20).sort_values().plot(kind='barh', ax=axes[0], color=COLORS['secondary'])
axes[0].set_title('Top 20 Missingness Rates')
axes[0].set_xlabel('Missing Fraction')

missingness['n_unique'].head(20).sort_values().plot(kind='barh', ax=axes[1], color=COLORS['primary'])
axes[1].set_title('Top 20 Feature Cardinalities')
axes[1].set_xlabel('Unique Values')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_quality_audit.png', dpi=150, bbox_inches='tight')
plt.show()

Quality report:


,metric,value
0,rows,177162
1,columns,434
2,duplicate_rows,0
3,target_missing,0
4,high_missing_columns,214
5,numeric_columns,403
6,categorical_columns,31


Top missing columns:


,missing_count,missing_pct,dtype,n_unique
id_24,175777,0.992182,float32,11
id_25,175653,0.991482,float32,204
id_21,175651,0.991471,float32,245
id_08,175650,0.991465,float32,84
id_07,175650,0.991465,float32,72
id_26,175646,0.991443,float32,67
id_22,175645,0.991437,float32,16
id_23,175645,0.991437,str,3
id_27,175645,0.991437,str,2
dist2,165973,0.936843,float32,1100


High-cardinality categorical columns:


,missing_count,missing_pct,dtype,n_unique


## 4. Target Variable Inspection

We inspect the fraud label distribution, class imbalance, and leakage-prone identifiers before any preprocessing decisions are made.

In [ ]:
if TARGET_COL in df.columns:
    target_counts = df[TARGET_COL].value_counts().sort_index()
    target_rate = df[TARGET_COL].mean()
    imbalance_ratio = (target_counts.iloc[0] / max(target_counts.iloc[1], 1)) if len(target_counts) > 1 else np.nan

    print(f'Target column: {TARGET_COL}')
    print(f'Fraud rate: {format_pct(target_rate)}')
    print(f'Imbalance ratio: 1:{imbalance_ratio:.1f}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    target_counts.plot(kind='bar', color=[COLORS['legit'], COLORS['fraud']], ax=axes[0], edgecolor='white')
    axes[0].set_title('Target Class Distribution')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Count')

    (target_counts / target_counts.sum()).plot(kind='pie', autopct='%.2f%%', ax=axes[1], colors=[COLORS['legit'], COLORS['fraud']])
    axes[1].set_ylabel('')
    axes[1].set_title('Target Class Share')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '02_target_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

leakage_candidates = [c for c in df.columns if c in {ID_COL, 'TransactionDT', TARGET_COL}]
print('Leakage-prone columns to exclude from modeling:')
print(leakage_candidates)

if 'TransactionAmt' in df.columns:
    fraud_amt = df.loc[df[TARGET_COL] == 1, 'TransactionAmt']
    legit_amt = df.loc[df[TARGET_COL] == 0, 'TransactionAmt']
    print('\nTransaction amount by class:')
    display(pd.DataFrame({
        'fraud_mean': [fraud_amt.mean()],
        'legit_mean': [legit_amt.mean()],
        'fraud_median': [fraud_amt.median()],
        'legit_median': [legit_amt.median()],
    }))

## 5. Univariate EDA: Numeric and Categorical Features

We summarize numerical distributions, skewness, variance, and missingness, then inspect categorical frequency patterns and rare labels.

In [3]:
# Numeric feature diagnostics
numeric_features = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL]
num_stats = []
for col in numeric_features:
    series = df[col].dropna()
    if len(series) == 0:
        continue
    num_stats.append({
        'feature': col,
        'missing_pct': df[col].isna().mean(),
        'mean': series.mean(),
        'median': series.median(),
        'std': series.std(),
        'skew': series.skew(),
        'kurtosis': series.kurtosis(),
        'variance': series.var(),
    })
num_stats_df = pd.DataFrame(num_stats).sort_values(['missing_pct', 'variance'], ascending=[False, False])
print('Top numeric features by missingness / variance:')
display(num_stats_df.head(20))

# Plot a compact set of numeric features
plot_numeric = num_stats_df.sort_values('variance', ascending=False).head(6)['feature'].tolist()
if 'TransactionAmt' in df.columns and 'TransactionAmt' not in plot_numeric:
    plot_numeric = ['TransactionAmt'] + plot_numeric[:5]

if len(plot_numeric) > 0:
    fig, axes = plt.subplots(len(plot_numeric), 2, figsize=(16, 4 * len(plot_numeric)))
    if len(plot_numeric) == 1:
        axes = np.array([axes])
    for i, col in enumerate(plot_numeric):
        series = df[col].dropna()
        sns.histplot(series, bins=50, kde=True, ax=axes[i, 0], color=COLORS['primary'])
        axes[i, 0].set_title(f'{col} distribution')
        sns.boxplot(x=series, ax=axes[i, 1], color=COLORS['accent'])
        axes[i, 1].set_title(f'{col} boxplot')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '03_numeric_eda.png', dpi=150, bbox_inches='tight')
    plt.show()

# Categorical feature diagnostics
categorical_features = [c for c in df.select_dtypes(include=['object']).columns if c != TARGET_COL]
cat_summary = []
for col in categorical_features:
    vc = df[col].fillna('Missing').value_counts(dropna=False)
    cat_summary.append({
        'feature': col,
        'missing_pct': df[col].isna().mean(),
        'n_unique': df[col].nunique(dropna=True),
        'top_category': vc.index[0] if len(vc) else None,
        'top_category_pct': vc.iloc[0] / len(df) if len(vc) else np.nan,
    })
cat_summary_df = pd.DataFrame(cat_summary).sort_values(['n_unique', 'missing_pct'], ascending=[False, False])
print('Top categorical features by cardinality:')
display(cat_summary_df.head(20))

plot_categorical = cat_summary_df.head(4)['feature'].tolist()
if len(plot_categorical) > 0:
    fig, axes = plt.subplots(len(plot_categorical), 1, figsize=(14, 4 * len(plot_categorical)))
    if len(plot_categorical) == 1:
        axes = [axes]
    for ax, col in zip(axes, plot_categorical):
        top_k = df[col].fillna('Missing').value_counts().head(10)
        top_k.plot(kind='bar', ax=ax, color=COLORS['secondary'])
        ax.set_title(f'{col} top values')
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '04_categorical_eda.png', dpi=150, bbox_inches='tight')
    plt.show()

Top numeric features by missingness / variance:


,feature,missing_pct,mean,median,std,skew,kurtosis,variance
398,id_24,0.992182,12.804332,11.000000,2.313877,1.132472,2.072322,5.354027
399,id_25,0.991482,331.502991,321.000000,99.840500,0.022380,-0.090813,9968.125977
396,id_21,0.991471,368.347443,252.000000,195.272736,1.234514,0.085737,38131.441406
386,id_08,0.991465,-37.945766,-34.000000,25.788679,-1.012426,0.774692,665.055969
385,id_07,0.991465,13.097222,13.500000,11.441351,-0.174544,0.622660,130.904510
400,id_26,0.991443,149.696564,150.000000,31.698189,-0.046527,-0.845565,1004.775208
397,id_22,0.991437,16.101517,14.000000,7.064178,3.134856,7.974208,49.902615
10,dist2,0.936843,233.421936,38.000000,526.724365,5.937070,60.783997,277438.531250
31,D7,0.933964,40.768612,0.000000,97.579887,2.926183,8.338670,9521.834961
393,id_18,0.923353,14.244569,15.000000,1.558001,2.016707,12.761763,2.427367


Top categorical features by cardinality:


,feature,missing_pct,n_unique,top_category,top_category_pct
30,DeviceInfo,0.798930,1259,Missing,0.798930
23,id_33,0.875419,164,Missing,0.875419
22,id_31,0.762410,116,Missing,0.762410
21,id_30,0.868047,72,Missing,0.868047
4,R_emaildomain,0.767586,60,Missing,0.767586
3,P_emaildomain,0.161524,59,gmail.com,0.385681
0,ProductCD,0.000000,5,W,0.744370
24,id_34,0.867765,4,Missing,0.867765
1,card4,0.002613,4,visa,0.650930
2,card6,0.002602,4,debit,0.745205


## 6. Bivariate EDA and Correlation Analysis

This section checks how the strongest features relate to the fraud target and inspects correlation among numeric variables to spot redundant signals.

In [ ]:
# Bivariate numeric vs target
if TARGET_COL in df.columns:
    numeric_top = num_stats_df.sort_values('variance', ascending=False).head(8)['feature'].tolist()
    if 'TransactionAmt' in df.columns and 'TransactionAmt' not in numeric_top:
        numeric_top = ['TransactionAmt'] + numeric_top[:7]

    bivariate_rows = []
    for col in numeric_top:
        legit = df.loc[df[TARGET_COL] == 0, col].dropna()
        fraud = df.loc[df[TARGET_COL] == 1, col].dropna()
        if len(legit) > 0 and len(fraud) > 0:
            effect = (fraud.mean() - legit.mean()) / (np.sqrt((fraud.var() + legit.var()) / 2) + 1e-9)
            bivariate_rows.append({
                'feature': col,
                'legit_mean': legit.mean(),
                'fraud_mean': fraud.mean(),
                'mean_diff': fraud.mean() - legit.mean(),
                'cohens_d': effect,
            })
    bivariate_df = pd.DataFrame(bivariate_rows).sort_values('cohens_d', ascending=False)
    print('Feature-target relationships:')
    display(bivariate_df)

    fig, axes = plt.subplots(len(numeric_top), 1, figsize=(14, 4 * len(numeric_top)))
    if len(numeric_top) == 1:
        axes = [axes]
    for ax, col in zip(axes, numeric_top):
        sns.kdeplot(df.loc[df[TARGET_COL] == 0, col].dropna(), ax=ax, label='Legit', fill=True, alpha=0.3, color=COLORS['legit'])
        sns.kdeplot(df.loc[df[TARGET_COL] == 1, col].dropna(), ax=ax, label='Fraud', fill=True, alpha=0.3, color=COLORS['fraud'])
        ax.set_title(f'{col} by Target')
        ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '05_bivariate_numeric_target.png', dpi=150, bbox_inches='tight')
    plt.show()

# Correlation analysis on numeric subset
corr_cols = num_stats_df.sort_values('missing_pct').head(20)['feature'].tolist()
if TARGET_COL in df.columns and TARGET_COL not in corr_cols:
    corr_cols = corr_cols[:19]
corr_df = df[corr_cols].copy()
corr_matrix = corr_df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, square=False)
plt.title('Correlation Heatmap for Selected Numeric Features')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

high_corr_pairs = []
for i, col1 in enumerate(corr_matrix.columns):
    for col2 in corr_matrix.columns[i+1:]:
        value = corr_matrix.loc[col1, col2]
        if abs(value) >= 0.90:
            high_corr_pairs.append({'feature_1': col1, 'feature_2': col2, 'correlation': value})
high_corr_df = pd.DataFrame(high_corr_pairs)
print('Highly correlated pairs (|r| >= 0.90):')
display(high_corr_df if not high_corr_df.empty else pd.DataFrame(columns=['feature_1', 'feature_2', 'correlation']))

## 7. Outlier Treatment, Imputation, Encoding, and Scaling

This section applies the cleaning and preprocessing decisions in a leakage-safe way. Training data drives the fitted transformations, and validation/test data are transformed with the same fitted objects.

In [ ]:
# Work on a copy so the EDA snapshot stays intact
clean_df = df.copy()

# Outlier treatment
clean_df = cap_outliers(clean_df)

# Missing values and categorical encoding are handled using project utilities
clean_df = handle_missing_values(clean_df)
clean_df, fitted_label_encoders = encode_categoricals(clean_df)

# Separate features and target after cleaning
if TARGET_COL in clean_df.columns:
    y = clean_df[TARGET_COL].astype(int)
    X = clean_df.drop(columns=[TARGET_COL])
else:
    raise ValueError(f'{TARGET_COL} is required for supervised preprocessing.')

# Keep a copy of the cleaned, encoded data before scaling
clean_df_snapshot = clean_df.copy()

print(f'Cleaned dataframe shape: {clean_df.shape}')
print(f'Encoded dataframe shape: {clean_df.shape}')
print(f'Categorical encoders fitted: {len(fitted_label_encoders)}')

# Scale numeric features only on the later split output; here we simply preserve the cleaned table
print('Cleaning complete. Full split/scaling is performed in the next section.')
display(clean_df.head(5))

## 8. Train/Validation/Test Split, Pipeline Assembly, and Artifact Saving

This section finishes the leakage-safe split, scales the features, saves the fitted preprocessing objects, and exports the datasets for the three model notebooks.

In [ ]:
# Re-load clean features for splitting and scaling
work_df = clean_df_snapshot.copy()
work_df = work_df.rename(columns={c: c for c in work_df.columns})

# Split data first to avoid leakage
X_train, X_val, X_test, y_train, y_val, y_test = split_data(work_df)

# Fit scaler on training split only
X_train_scaled, X_val_scaled, X_test_scaled, scaler = scale_features(X_train, X_val, X_test)

# Run the project preprocessing pipeline as the canonical artifact builder
preprocess_artifacts = run_preprocessing_pipeline(df.copy())

# Persist the final split outputs and fitted preprocessing objects
artifact_bundle = {
    'X_train': X_train_scaled,
    'X_val': X_val_scaled,
    'X_test': X_test_scaled,
    'y_train': y_train,
    'y_val': y_val,
    'y_test': y_test,
    'scaler': scaler,
    'label_encoders': fitted_label_encoders,
    'feature_names': list(X_train_scaled.columns),
}

with open(MODEL_DIR / 'preprocessed_data.pkl', 'wb') as f:
    pickle.dump(artifact_bundle, f)

joblib.dump(scaler, SCALER_PATH)
joblib.dump(fitted_label_encoders, LABEL_ENCODERS_PATH)
joblib.dump(list(X_train_scaled.columns), FEATURE_NAMES_PATH)

print('Saved preprocessing artifacts:')
print(f'  {MODEL_DIR / "preprocessed_data.pkl"}')
print(f'  {SCALER_PATH}')
print(f'  {LABEL_ENCODERS_PATH}')
print(f'  {FEATURE_NAMES_PATH}')
print('\nSplit shapes:')
print(f'  X_train: {X_train_scaled.shape}')
print(f'  X_val:   {X_val_scaled.shape}')
print(f'  X_test:  {X_test_scaled.shape}')

## 9. Validation Tests and Reproducibility Checks

The final section verifies that the saved artifacts are consistent, the transformed matrices are finite, and the outputs are deterministic under the fixed seed.

In [ ]:
# Load persisted artifacts and run quick smoke tests
with open(MODEL_OUT_DIR / 'preprocessed_data.pkl', 'rb') as f:
    saved_bundle = pickle.load(f)

assert saved_bundle['X_train'].shape[0] == len(saved_bundle['y_train'])
assert saved_bundle['X_val'].shape[0] == len(saved_bundle['y_val'])
assert saved_bundle['X_test'].shape[0] == len(saved_bundle['y_test'])

for name in ['X_train', 'X_val', 'X_test']:
    matrix = saved_bundle[name]
    assert np.isfinite(matrix.to_numpy()).all(), f'{name} contains non-finite values'

print('Smoke tests passed:')
print(f"  Train matrix shape: {saved_bundle['X_train'].shape}")
print(f"  Val matrix shape:   {saved_bundle['X_val'].shape}")
print(f"  Test matrix shape:  {saved_bundle['X_test'].shape}")
print(f"  Feature count:      {len(saved_bundle['feature_names'])}")
print(f"  Label encoder count:{len(saved_bundle['label_encoders'])}")

summary_table = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(saved_bundle['X_train']), len(saved_bundle['X_val']), len(saved_bundle['X_test'])],
    'fraud_rate': [saved_bundle['y_train'].mean(), saved_bundle['y_val'].mean(), saved_bundle['y_test'].mean()],
})
print('\nSplit summary:')
display(summary_table)

## 2. Standardize Column Names and Data Types

We normalize column names, inspect the key data types, and perform safe conversions where needed. This keeps downstream preprocessing predictable and easier to debug.

In [ ]:
def normalize_column_name(name: str) -> str:
    cleaned = name.strip().replace(' ', '_').replace('-', '_')
    cleaned = ''.join(ch if ch.isalnum() or ch == '_' else '_' for ch in cleaned)
    while '__' in cleaned:
        cleaned = cleaned.replace('__', '_')
    return cleaned.lower()

# Keep the original schema for compatibility with the project pipeline.
# We only generate a normalization preview here for documentation purposes.
original_columns = list(df.columns)
normalized_preview = pd.DataFrame({
    'original': original_columns,
    'normalized_preview': [normalize_column_name(c) for c in original_columns],
})
print('Column name normalization preview:')
display(normalized_preview.head(20))

# Data type inspection with the original column names preserved
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
datetime_cols = [c for c in df.columns if 'Date' in c or 'Time' in c or c.endswith('DT')]

print(f'Numeric columns: {len(numeric_cols)}')
print(f'Categorical columns: {len(categorical_cols)}')
print(f'Datetime-like columns: {len(datetime_cols)}')

# Lightweight conversions for known fields
if 'TransactionDT' in df.columns:
    df['TransactionDT'] = pd.to_numeric(df['TransactionDT'], errors='coerce')
if TARGET_COL in df.columns:
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce').astype('Int64')

conversion_summary = pd.DataFrame({
    'column': ['TransactionDT', TARGET_COL],
    'final_dtype': [str(df[c].dtype) if c in df.columns else 'missing' for c in ['TransactionDT', TARGET_COL]],
})
display(conversion_summary)